# Model Comparison & Portfolio Allocation
**Tasks 4, 5, 6: Analysis → Allocation → Comparison**

This notebook:
1. Runs volatility & trend analysis (Task 4)
2. Loads all model forecasts and compares them (Task 6)
3. Selects the best model and builds the portfolio (Task 5)
4. Outputs the final ₹10,00,000 allocation table for StockGro trading

In [ ]:
import sys
sys.path.append("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.fetch_data import load_all_raw, STOCK_UNIVERSE, SECTOR_MAP
from src.data.preprocess import preprocess_all
from src.data.screening import sector_momentum
from src.analysis.volatility import volatility_summary
from src.analysis.trend_analysis import trend_summary
from src.analysis.correlation import return_correlation_matrix, correlation_weights, low_correlation_pairs
from src.portfolio.allocation import build_portfolio

plt.style.use("seaborn-v0_8-whitegrid")
print("Imports OK")

In [ ]:
raw = load_all_raw()
processed = preprocess_all(raw, save=False)
print(f"Loaded {len(processed)} stocks")

---
## Task 4: Volatility & Trend Analysis

In [ ]:
vol_df = volatility_summary(raw)
print("── Volatility Summary ──")
print(vol_df.to_string())

In [ ]:
# Volatility bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vol_df["Ann_RollingVol"].sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Annualised Rolling Volatility")
axes[0].set_xlabel("Volatility")

vol_df["GARCH_FcstVol"].sort_values().plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("GARCH Forecast Volatility")
axes[1].set_xlabel("Volatility")

plt.tight_layout()
plt.show()

In [ ]:
trend_df = trend_summary(raw)
print("── Trend Summary ──")
print(trend_df.to_string())

In [ ]:
# Correlation matrix
corr_df = return_correlation_matrix(raw)
corr_w = correlation_weights(corr_df)

fig, ax = plt.subplots(figsize=(10, 8))
labels = [t.replace('.NS', '') for t in corr_df.index]
sns.heatmap(corr_df.values, annot=True, fmt=".2f",
            xticklabels=labels, yticklabels=labels,
            cmap="RdBu_r", center=0, ax=ax)
ax.set_title("6-Month Return Correlation Matrix")
plt.tight_layout()
plt.show()

lcp = low_correlation_pairs(corr_df)
print("\nLow-Correlation Pairs (|r| < 0.3):")
print(lcp.to_string())

---
## Task 6: Model Comparison

Load the combined metrics file generated by individual model notebooks or `main_pipeline.py`.

In [ ]:
from pathlib import Path

mc_path = Path("..") / "results" / "metrics" / "model_comparison.csv"

if mc_path.exists():
    mc = pd.read_csv(mc_path)
    print("── Full Model Comparison ──")
    print(mc.to_string())
else:
    print("model_comparison.csv not found.")
    print("Run individual model notebooks first, or run:")
    print('  python main_pipeline.py --task 3')
    mc = None

In [ ]:
if mc is not None:
    # Average metrics per model
    avg = mc.groupby("Model")[["RMSE", "MAPE", "DirAcc"]].mean().round(4)
    print("\n── Average Metrics by Model ──")
    print(avg.to_string())
    print(f"\nBest model by MAPE: {avg['MAPE'].idxmin()}")
    print(f"Best model by DirAcc: {avg['DirAcc'].idxmax()}")

    # Comparison bar charts
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for i, metric in enumerate(["MAPE", "RMSE", "DirAcc"]):
        ax = axes[i]
        mc.pivot_table(values=metric, index="Ticker", columns="Model").plot(
            kind="bar", ax=ax, rot=30
        )
        ax.set_title(metric)
        ax.legend(fontsize=7)
        ax.tick_params(labelsize=8)
    plt.suptitle("Model Comparison Across Stocks", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Avg comparison
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    colors = ["#4A90D9", "#F5A623", "#7B68EE", "#20B2AA", "#E74C3C"]
    for i, metric in enumerate(["MAPE", "RMSE", "DirAcc"]):
        avg[metric].plot(kind="bar", ax=axes[i], color=colors[:len(avg)])
        axes[i].set_title(f"Avg {metric}")
        axes[i].tick_params(axis="x", rotation=30)
    plt.suptitle("Average Model Performance", fontsize=14)
    plt.tight_layout()
    plt.show()

---
## Task 5: Portfolio Allocation

Using the best model's forecasts + volatility + correlation analysis to allocate ₹10,00,000.

In [ ]:
# Load the best model's future forecasts
# (Choose based on Task 6 results above — default to ARIMA)

from src.models.arima import run_arima_pipeline

arima_preds, arima_fc, arima_met = run_arima_pipeline(processed, n_forecast=5)

# Current prices
current_prices = {}
for ticker, df in raw.items():
    close = df["Close"].squeeze().dropna()
    current_prices[ticker] = float(close.iloc[-1])

# Volatility estimates
vol_estimates = vol_df["GARCH_FcstVol"].to_dict()

# Sector momentum
sec_mom = sector_momentum(raw, SECTOR_MAP)

tickers = list(STOCK_UNIVERSE.keys())

alloc = build_portfolio(
    tickers=tickers,
    sector_map=SECTOR_MAP,
    stock_names=STOCK_UNIVERSE,
    future_forecasts=arima_fc,
    current_prices=current_prices,
    volatility_estimates=vol_estimates,
    corr_weights=corr_w,
    sector_returns=sec_mom,
)

print("\n── Final Portfolio Allocation ──")
print(alloc.to_string())
print(f"\nTotal Deployed: ₹{alloc['Amount'].sum():,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

alloc_sorted = alloc.sort_values("Weight", ascending=False)
axes[0].pie(alloc_sorted["Weight"], labels=alloc_sorted["Name"],
            autopct="%1.1f%%", pctdistance=0.85)
axes[0].set_title("Allocation by Stock")

sector_alloc = alloc.groupby("Sector")["Amount"].sum().sort_values(ascending=False)
axes[1].bar(sector_alloc.index, sector_alloc.values, color=["#4A90D9", "#F5A623", "#2ECC71", "#E74C3C", "#9B59B6"])
axes[1].set_title("Allocation by Sector (₹)")
axes[1].set_ylabel("Amount (₹)")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Portfolio Allocation — ₹10,00,000", fontsize=14)
plt.tight_layout()
plt.show()

## StockGro Execution Summary

Use the allocation table above to place trades on StockGro:

1. Open StockGro app → Trackers → "Portfolio - Time Series Analysis 2026"
2. Buy each stock at the approximate number of shares shown in `Shares_Approx`
3. Execute before market close (3:25 PM) on your chosen Day 1
4. Record closing prices at market close of Day 2
5. Use the Task 8 notebook or dashboard to compare predicted vs actual